# 05 - Recomendaciones LLM para retención de clientes

Objetivo: generar estrategias personalizadas de retención usando IA generativa a partir de:

- Probabilidad estimada de churn (`score_churn`)
- Predicción del modelo (`pred_churn`)
- Segmento comercial generado en el notebook 04
- Variables de comportamiento del cliente

Este notebook usa directamente los segmentos del notebook 04: **Estratégicos, Recurrentes, Promocionales y En riesgo**. No se realiza mapeo a otros nombres para evitar inconsistencias.

## 1. Instalación y librerías

In [ ]:
# Ejecutar solo si estás en Colab o si no tienes estas librerías instaladas
# !pip install -q huggingface_hub python-dotenv

In [23]:
!pip install google-generativeai

     ---------------------------------------- 0.0/108.8 kB ? eta -:--:--
     -------------- ------------------------ 41.0/108.8 kB 1.9 MB/s eta 0:00:01
     -------------------------------------- 108.8/108.8 kB 1.6 MB/s eta 0:00:00
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/155.1 kB ? eta -:--:--
   ---------------------------------------- 155.1/155.1 kB 4.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------ --------------------------------- 0.2/1.3 MB 6.3 MB/s eta 0:00:01
   -----------------


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import os
import json
import ast
import re

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

## 2. Cargar dataset final

El archivo recomendado es `clientes_churn_final.csv`, porque contiene tanto las predicciones de churn como los segmentos del notebook 04.

In [2]:
# Ajusta esta ruta si estás trabajando en Google Colab
# Ejemplo Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = Path('/content/drive/MyDrive/PI_LV/data/processed/clientes_churn_final.csv')

DATA_PATH = Path('../data/processed/clientes_churn_final.csv')

if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/clientes_churn_final.csv')

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(80355, 29)


,cliente_id_new,fecha_primera_compra,fecha_ultima_compra,recency,frequency,monetary,tenure_days,active_month_ratio,avg_ticket,std_ticket,...,dias_promedio_vencimiento,flag_credito,churn_60,churn_70,churn_90,churn,score_churn,pred_churn,cluster,segmento
0,1027461,2024-05-08,2025-02-28,307,5,1970.18,296,0.300000,394.03600,256.588208,...,0.0,0.0,1,1,1,1,0.688902,1,0,En riesgo
1,1089134,2025-02-06,2025-12-31,1,20,17891.83,328,1.000000,894.59150,687.827867,...,0.0,0.0,0,0,0,0,0.144226,0,3,Estratégicos
2,1089187,2024-02-19,2025-12-08,24,31,7418.34,658,0.782609,239.30129,180.339686,...,0.0,0.0,0,0,0,0,0.442597,1,3,Estratégicos
3,1111859,2024-10-16,2025-12-18,14,20,3583.75,428,0.866667,179.18750,99.518539,...,0.0,0.0,0,0,0,0,0.201553,0,3,Estratégicos
4,1133970,2024-04-08,2025-12-04,28,8,937.10,605,0.333333,117.13750,72.922652,...,0.0,0.0,0,0,0,0,0.744322,1,0,En riesgo


## 3. Verificación de columnas clave

In [3]:
COLUMNAS_REQUERIDAS = [
    'cliente_id_new',
    'segmento',
    'score_churn',
    'pred_churn',
    'recency',
    'frequency',
    'monetary'
]

faltantes = [col for col in COLUMNAS_REQUERIDAS if col not in df.columns]

if faltantes:
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')

df[COLUMNAS_REQUERIDAS].head()

,cliente_id_new,segmento,score_churn,pred_churn,recency,frequency,monetary
0,1027461,En riesgo,0.688902,1,307,5,1970.18
1,1089134,Estratégicos,0.144226,0,1,20,17891.83
2,1089187,Estratégicos,0.442597,1,24,31,7418.34
3,1111859,Estratégicos,0.201553,0,14,20,3583.75
4,1133970,En riesgo,0.744322,1,28,8,937.10


In [4]:
SEGMENTOS_VALIDOS = ['Estratégicos', 'Recurrentes', 'Promocionales', 'En riesgo']

print('Segmentos encontrados:')
print(df['segmento'].value_counts(dropna=False))

segmentos_no_esperados = sorted(set(df['segmento'].dropna()) - set(SEGMENTOS_VALIDOS))

if segmentos_no_esperados:
    print('Advertencia: hay segmentos no esperados:', segmentos_no_esperados)
else:
    print('Todos los segmentos coinciden con el notebook 04.')

Segmentos encontrados:
segmento
Estratégicos     32137
En riesgo        19491
Recurrentes      16171
Promocionales    12556
Name: count, dtype: int64
Todos los segmentos coinciden con el notebook 04.


## 4. Reglas de negocio para prioridad y canal

Estas reglas usan directamente `score_churn` y `segmento`. No se usa `segmento_prompt`.

In [5]:
PRIORIDADES_VALIDAS = [
    'Baja - Mantener relación',
    'Media - Monitorear comportamiento',
    'Media - Incentivar recompra',
    'Alta - Contacto personalizado',
    'Crítica - Recuperación inmediata'
]

CANALES_VALIDOS = ['WhatsApp', 'Teléfono', 'Email', 'Visita comercial']

def safe_value(row, col, default='No disponible'):
    return row[col] if col in row.index and pd.notna(row[col]) else default

def formato_monto(valor):
    if valor == 'No disponible':
        return valor
    return f'S/ {float(valor):,.2f}'

def formato_pct(valor):
    if valor == 'No disponible':
        return valor
    valor = float(valor)
    if abs(valor) <= 1:
        return f'{valor:.1%}'
    return f'{valor:.1f}%'

def asignar_nivel_riesgo_score(score):
    score = float(score)
    if score >= 0.75:
        return 'Crítico'
    elif score >= 0.50:
        return 'Alto'
    elif score >= 0.30:
        return 'Medio'
    else:
        return 'Bajo'

def asignar_prioridad_prompt(row):
    score = float(safe_value(row, 'score_churn', 0))
    segmento = safe_value(row, 'segmento', '')

    if score >= 0.75:
        return 'Crítica - Recuperación inmediata'

    elif score >= 0.50:
        if segmento in ['Estratégicos', 'En riesgo']:
            return 'Alta - Contacto personalizado'
        return 'Media - Incentivar recompra'

    elif score >= 0.30:
        if segmento == 'Promocionales':
            return 'Media - Incentivar recompra'
        return 'Media - Monitorear comportamiento'

    else:
        return 'Baja - Mantener relación'

def sugerir_canal(prioridad):
    if prioridad == 'Crítica - Recuperación inmediata':
        return 'Teléfono'
    elif prioridad == 'Alta - Contacto personalizado':
        return 'WhatsApp'
    elif prioridad in ['Media - Monitorear comportamiento', 'Media - Incentivar recompra']:
        return 'WhatsApp'
    else:
        return 'Email'

# Variables finales para el prompt
df['nivel_riesgo_prompt'] = df['score_churn'].apply(asignar_nivel_riesgo_score)
df['prioridad_prompt'] = df.apply(asignar_prioridad_prompt, axis=1)
df['canal_prompt'] = df['prioridad_prompt'].apply(sugerir_canal)

df[['cliente_id_new', 'segmento', 'score_churn', 'pred_churn', 'nivel_riesgo_prompt', 'prioridad_prompt', 'canal_prompt']].head()

,cliente_id_new,segmento,score_churn,pred_churn,nivel_riesgo_prompt,prioridad_prompt,canal_prompt
0,1027461,En riesgo,0.688902,1,Alto,Alta - Contacto personalizado,WhatsApp
1,1089134,Estratégicos,0.144226,0,Bajo,Baja - Mantener relación,Email
2,1089187,Estratégicos,0.442597,1,Medio,Media - Monitorear comportamiento,WhatsApp
3,1111859,Estratégicos,0.201553,0,Bajo,Baja - Mantener relación,Email
4,1133970,En riesgo,0.744322,1,Alto,Alta - Contacto personalizado,WhatsApp


## 5. Construcción del contexto del cliente

In [6]:
def build_customer_context(row):
    segmento = safe_value(row, 'segmento')
    score = safe_value(row, 'score_churn')
    pred = safe_value(row, 'pred_churn')
    prioridad = safe_value(row, 'prioridad_prompt')
    canal = safe_value(row, 'canal_prompt')
    nivel = safe_value(row, 'nivel_riesgo_prompt')

    pred_texto = 'No disponible'
    if pred != 'No disponible':
        pred_texto = 'Sí' if int(pred) == 1 else 'No'

    flag_credito = safe_value(row, 'flag_credito', 'No disponible')
    credito_texto = 'No disponible'
    if flag_credito != 'No disponible':
        credito_texto = 'Sí' if int(flag_credito) == 1 else 'No'

    return f'''
Cliente con las siguientes características comerciales:
- ID cliente: {safe_value(row, 'cliente_id_new')}
- Segmento comercial: {segmento}
- Probabilidad estimada de churn: {formato_pct(score)}
- Predicción de churn del modelo: {pred_texto}
- Nivel de riesgo asignado: {nivel}
- Prioridad sugerida por reglas internas: {prioridad}
- Canal preliminar sugerido: {canal}
- Días desde la última compra: {safe_value(row, 'recency')}
- Frecuencia total de compra: {safe_value(row, 'frequency')}
- Monto total comprado: {formato_monto(safe_value(row, 'monetary'))}
- Ticket promedio: {formato_monto(safe_value(row, 'avg_ticket'))}
- Compras en los últimos 70 días: {safe_value(row, 'compras_70d')}
- Gasto en los últimos 70 días: {formato_monto(safe_value(row, 'gasto_70d'))}
- Tendencia reciente de gasto: {safe_value(row, 'spend_trend_70')}
- Descuento promedio aplicado: {formato_pct(safe_value(row, 'descuento_porc_promedio'))}
- Margen promedio: {formato_pct(safe_value(row, 'margen_pct'))}
- Cliente con crédito: {credito_texto}
'''.strip()

## 6. Prompt final para el LLM

Este prompt está alineado con los segmentos reales del notebook 04.

In [7]:
def build_prompt(context):
    return f"""
Actúa como un Gerente Senior de Retención de Clientes en una empresa retail.

Tu objetivo es crear una estrategia personalizada de retención para un cliente, usando su comportamiento de compra, segmento y probabilidad de churn.

CONTEXTO DEL CLIENTE:
{context}

REGLAS DE NEGOCIO:

Los segmentos de cliente pueden ser:
1. Estratégicos: clientes de alto valor y relevancia para el negocio.
2. Recurrentes: clientes que compran con frecuencia pero pueden estar reduciendo su actividad.
3. Promocionales: clientes sensibles a descuentos o promociones.
4. En riesgo: clientes con alta probabilidad de abandono.

Los niveles de prioridad son:
1. Baja - Mantener relación
2. Media - Monitorear comportamiento
3. Media - Incentivar recompra
4. Alta - Contacto personalizado
5. Crítica - Recuperación inmediata

INTERPRETACIÓN OBLIGATORIA:

- Si la probabilidad de churn es mayor o igual a 75%, usa prioridad \"Crítica - Recuperación inmediata\".
- Si la probabilidad de churn está entre 50% y 75%, usa prioridad \"Alta - Contacto personalizado\" o \"Media - Incentivar recompra\" según el segmento.
- Si la probabilidad de churn está entre 30% y 50%, usa prioridad \"Media - Monitorear comportamiento\" o \"Media - Incentivar recompra\" según el segmento.
- Si la probabilidad de churn es menor a 30%, usa prioridad \"Baja - Mantener relación\".

- Si el cliente es Estratégico, prioriza retención personalizada y beneficios exclusivos.
- Si es Recurrente, enfócate en reactivar frecuencia de compra.
- Si es Promocional, usa incentivos económicos concretos.
- Si está En riesgo, prioriza acciones inmediatas de recuperación.

- El canal debe ser coherente con la urgencia:
  - Alta o Crítica: Teléfono o WhatsApp.
  - Media: WhatsApp o Email.
  - Baja: Email.

SALIDA REQUERIDA:

Devuelve SOLAMENTE un objeto JSON válido, sin markdown, sin HTML y sin texto adicional.

El JSON debe tener EXACTAMENTE esta estructura:

{{
  \"analisis_breve\": \"Una frase breve explicando el comportamiento comercial del cliente, sin repetir los datos numéricos literalmente.\",
  \"prioridad\": \"Baja - Mantener relación | Media - Monitorear comportamiento | Media - Incentivar recompra | Alta - Contacto personalizado | Crítica - Recuperación inmediata\",
  \"canal_sugerido\": \"WhatsApp | Teléfono | Email | Visita comercial\",
  \"incentivo_recomendado\": \"Ejemplo concreto de incentivo comercial\",
  \"estrategias\": {{
    \"accion_1\": \"Primera acción concreta de retención en español.\",
    \"accion_2\": \"Segunda acción concreta de retención en español.\",
    \"accion_3\": \"Tercera acción concreta de retención en español.\",
    \"accion_4\": \"Cuarta acción concreta de retención en español.\"
  }}
}}

REGLAS OBLIGATORIAS:

- Todo debe estar en español.
- No uses inglés.
- No uses HTML.
- No uses markdown.
- No generes texto fuera del JSON.
- Deben existir exactamente accion_1, accion_2, accion_3 y accion_4.
- Las acciones deben ser distintas entre sí.
- No uses frases genéricas como \"revisar el perfil del cliente\".
- No inventes datos que no estén en el contexto.
"""

## 7. Conexión con Hugging Face y Gemini

Hugging Face

In [ ]:
load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if not hf_token:
    print('Advertencia: no se encontró HF_TOKEN en el entorno. Configúralo antes de llamar al modelo.')

client = InferenceClient(
    model='mistralai/Mistral-7B-Instruct-v0.2',
    provider='featherless-ai',
    token=hf_token
)

Gemini

In [41]:
import google.generativeai as genai

gemini_key = os.getenv("GEMINI_API_KEY")

if gemini_key:
    genai.configure(api_key=gemini_key)
    gemini_model = genai.GenerativeModel("gemini-2.5-flash")
else:
    gemini_model = None

In [38]:
from dotenv import load_dotenv
import os

load_dotenv()

print("HF_TOKEN:", "OK" if os.getenv("HF_TOKEN") else "NO")
print("GEMINI_API_KEY:", "OK" if os.getenv("GEMINI_API_KEY") else "NO")

HF_TOKEN: OK
GEMINI_API_KEY: OK


## 8. Generar, limpiar y validar respuesta JSON

In [42]:
import time
import json
import re
import google.generativeai as genai
from huggingface_hub.utils import HfHubHTTPError

def generate_strategy_hf(prompt, max_retries=1, wait_seconds=3):
    last_error = None

    for intento in range(1, max_retries + 1):
        try:
            response = client.chat_completion(
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                max_tokens=800,
                temperature=0.4
            )

            return response.choices[0].message["content"]

        except HfHubHTTPError as e:
            last_error = e
            print(f"Hugging Face falló en intento {intento}: {e}")
            time.sleep(wait_seconds)

        except Exception as e:
            last_error = e
            print(f"Error inesperado en Hugging Face intento {intento}: {e}")
            time.sleep(wait_seconds)

    raise RuntimeError(f"Hugging Face no respondió correctamente: {last_error}")


def generate_strategy_gemini(prompt):
    if gemini_model is None:
        raise ValueError("Gemini API Key no configurada.")

    response = gemini_model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.4,
            "max_output_tokens": 800,
            "response_mime_type": "application/json"
        }
    )

    return response.text


def generate_strategy_llm(prompt):
    try:
        return generate_strategy_hf(
            prompt,
            max_retries=1,
            wait_seconds=3
        )
    except Exception as e:
        print(f"Hugging Face no disponible. Se usará Gemini. Error: {e}")

    try:
        return generate_strategy_gemini(prompt)
    except Exception as e:
        print(f"Gemini también falló. Error: {e}")

    return None


def extract_json(text):
    texto = str(text).strip()
    texto = texto.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", texto, flags=re.DOTALL)
        if not match:
            raise ValueError("No se encontró un objeto JSON en la respuesta del modelo.")
        return json.loads(match.group(0))


def validate_strategy_json(data):
    required_keys = {
        "analisis_breve",
        "prioridad",
        "canal_sugerido",
        "incentivo_recomendado",
        "estrategias"
    }

    if not isinstance(data, dict):
        raise ValueError("La respuesta no es un diccionario JSON.")

    if set(data.keys()) != required_keys:
        raise ValueError(f"El JSON no tiene exactamente las claves requeridas: {data.keys()}")

    if data["prioridad"] not in PRIORIDADES_VALIDAS:
        raise ValueError(f'Prioridad no válida: {data["prioridad"]}')

    if data["canal_sugerido"] not in CANALES_VALIDOS:
        raise ValueError(f'Canal no válido: {data["canal_sugerido"]}')

    acciones = data["estrategias"]
    acciones_requeridas = {
        "accion_1",
        "accion_2",
        "accion_3",
        "accion_4"
    }

    if not isinstance(acciones, dict):
        raise ValueError("El campo estrategias debe ser un diccionario.")

    if set(acciones.keys()) != acciones_requeridas:
        raise ValueError(f"Las acciones no tienen exactamente accion_1 a accion_4: {acciones.keys()}")

    valores = [
        str(acciones[k]).strip()
        for k in ["accion_1", "accion_2", "accion_3", "accion_4"]
    ]

    if any(not v for v in valores):
        raise ValueError("Hay acciones vacías.")

    if len(set(v.lower() for v in valores)) != 4:
        raise ValueError("Las cuatro acciones deben ser distintas entre sí.")

    texto_total = json.dumps(data, ensure_ascii=False).lower()

    palabras_invalidas = [
        "another strategy",
        "could be effective",
        "<div",
        "</div>",
        "<br",
        "revisar el perfil del cliente"
    ]

    if any(p in texto_total for p in palabras_invalidas):
        raise ValueError("La respuesta contiene frases o formatos no permitidos.")

    return True


def generate_strategy_json(row, max_retries=2):
    context = build_customer_context(row)
    prompt = build_prompt(context)

    last_error = None

    for intento in range(max_retries + 1):
        raw = generate_strategy_llm(prompt)

        if raw is None:
            raise ValueError("No se pudo generar respuesta con Hugging Face ni Gemini.")

        try:
            data = extract_json(raw)
            validate_strategy_json(data)
            return data

        except Exception as e:
            last_error = e
            prompt = prompt + """

La respuesta anterior no cumplió las reglas.
Vuelve a responder SOLO con JSON válido.
Debe respetar exactamente esta estructura: analisis_breve, prioridad, canal_sugerido, incentivo_recomendado y estrategias.
El campo estrategias debe tener exactamente accion_1, accion_2, accion_3 y accion_4.
Todo debe estar en español.
No uses inglés.
No uses HTML.
No uses markdown.
No uses placeholders.
"""

    raise ValueError(f"No se pudo generar una estrategia válida. Último error: {last_error}")

## 9. Prueba con un cliente

In [33]:
row = df.sample(1, random_state=42).iloc[0]
context = build_customer_context(row)
prompt = build_prompt(context)

print(context)

Cliente con las siguientes características comerciales:
- ID cliente: C136344
- Segmento comercial: Estratégicos
- Probabilidad estimada de churn: 32.8%
- Predicción de churn del modelo: No
- Nivel de riesgo asignado: Medio
- Prioridad sugerida por reglas internas: Media - Monitorear comportamiento
- Canal preliminar sugerido: WhatsApp
- Días desde la última compra: 15
- Frecuencia total de compra: 29
- Monto total comprado: S/ 15,042.43
- Ticket promedio: S/ 518.70
- Compras en los últimos 70 días: 1
- Gasto en los últimos 70 días: S/ 369.08
- Tendencia reciente de gasto: -0.4609569587065763
- Descuento promedio aplicado: 9.5%
- Margen promedio: 55.1%
- Cliente con crédito: No


In [34]:
# Ejecuta esta celda cuando tengas HF_TOKEN configurado
strategy_json = generate_strategy_json(row)
print(json.dumps(strategy_json, ensure_ascii=False, indent=2))

{
  "analisis_breve": "El cliente C136344 pertenece al segmento comercial de Estratégicos. Su probabilidad estimada de churn es del 32.8%. La última compra del cliente fue 15 días atrás, y su frecuencia total de compra es de 29. La cantidad total comprada por el cliente es de S/ 15,042.43.",
  "prioridad": "Media - Monitorear comportamiento",
  "canal_sugerido": "WhatsApp",
  "incentivo_recomendado": "Ofrecerle un descuento especial por su lealtad al negocio.",
  "estrategias": {
    "accion_1": "Contactarlo por WhatsApp para ofrecerle el descuento especial.",
    "accion_2": "Revisar su historial de compras para identificar productos que han tenido una baja demanda recientemente. Si se identifican productos que caen en esta categoría, considerar ofrecerle un descuento especial en esos productos.",
    "accion_3": "Revisar su historial de compras para identificar productos que han tenido una alta demanda recientemente. Si se identifican productos que caen en esta categoría, considerar 

## 10. Aplicar a una muestra y crear tabla de estrategias

In [43]:
df_sample = df.head(5).copy()

estrategias = []

for idx, row in df_sample.iterrows():
    try:
        estrategia = generate_strategy_json(row)
    except Exception as e:
        print(f"Error en cliente {row['cliente_id_new']}: {e}")
        estrategia = {
            "analisis_breve": "No se pudo generar la estrategia por un error temporal del servicio LLM.",
            "prioridad": row["prioridad_prompt"],
            "canal_sugerido": row["canal_prompt"],
            "incentivo_recomendado": "Pendiente de generación.",
            "estrategias": {
                "accion_1": "Reintentar la generación de estrategia para este cliente.",
                "accion_2": "Mantener seguimiento comercial según su nivel de riesgo.",
                "accion_3": "Aplicar la prioridad sugerida por las reglas internas.",
                "accion_4": "Registrar el caso para evaluación posterior."
            }
        }

    estrategias.append(estrategia)

df_sample["estrategia_json"] = estrategias
df_sample["estrategia_texto"] = df_sample["estrategia_json"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)

Hugging Face falló en intento 1: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/featherless-ai/v1/chat/completions' (Amz CF ID: oCshqAAS9n9uoY4fYT35-LP-4GcsbQmJZupOSeh91x6QEyctXyDWVA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504
Hugging Face no disponible. Se usará Gemini. Error: Hugging Face no respondió correctamente: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/featherless-ai/v1/chat/completions' (Amz CF ID: oCshqAAS9n9uoY4fYT35-LP-4GcsbQmJZupOSeh91x6QEyctXyDWVA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504
Hugging Face falló en intento 1: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/featherless-ai/v1/chat/completions' (Amz CF ID: vmqdFhTiPwsR60XP22bwuSR1e5c3TiCHsxfJxysa5vGV9sN3OXN9wg==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504
Hugging Face no disponible. Se 

In [50]:
df_estrategias = pd.concat(
    [
        df_sample[[
            "cliente_id_new",
            "segmento",
            "score_churn",
            "pred_churn",
            "nivel_riesgo_prompt",
            "prioridad_prompt",
            "canal_prompt"
        ]].reset_index(drop=True),
        pd.json_normalize(df_sample["estrategia_json"]).reset_index(drop=True)
    ],
    axis=1
)

df_estrategias

,cliente_id_new,segmento,score_churn,pred_churn,nivel_riesgo_prompt,prioridad_prompt,canal_prompt,analisis_breve,prioridad,canal_sugerido,incentivo_recomendado,estrategias.accion_1,estrategias.accion_2,estrategias.accion_3,estrategias.accion_4
0,1027461,En riesgo,0.688902,1,Alto,Alta - Contacto personalizado,WhatsApp,El cliente con ID 1027461 está en el segmento ...,Alta - Contacto personalizado,WhatsApp,"Ofrecer un descuento exclusivo por WhatsApp, p...",Contactar al cliente por WhatsApp para ofrecer...,Enviar un correo electrónico personalizado al ...,Realizar una llamada telefónica al cliente par...,Enviar una carta de correo postal personalizad...
1,1089134,Estratégicos,0.144226,0,Bajo,Baja - Mantener relación,Email,El cliente con ID 1089134 pertenece al segment...,Baja - Mantener relación,Email,No aplicable,Enviar un correo electrónico personalizado al ...,Realizar una llamada telefónica al cliente par...,Enviar un correo electrónico personalizado al ...,Realizar una visita comercial al cliente en su...
2,1089187,Estratégicos,0.442597,1,Medio,Media - Monitorear comportamiento,WhatsApp,"El cliente 1089187, segmento estratégico, ha p...",Media - Monitorear comportamiento,WhatsApp,Envío personalizado de un cupón de descuento p...,Contactarlo a través de WhatsApp para ofrecerl...,Enviarle un correo electrónico con información...,Realizar una llamada telefónica para ofrecerle...,Enviarle una notificación push de WhatsApp con...
3,1111859,Estratégicos,0.201553,0,Bajo,Baja - Mantener relación,Email,El cliente con ID 1111859 pertenece al segment...,Baja - Mantener relación,Email,Envío de ofertas especiales periódicas por cor...,Envío de una carta de agradecimiento personali...,Envío de una oferta especial por correo electr...,Envío de una oferta de regalo por correo elect...,Envío de una oferta de membresía por correo el...
4,1133970,En riesgo,0.744322,1,Alto,Alta - Contacto personalizado,WhatsApp,El cliente 1133970 pertenece al segmento de ri...,Alta - Contacto personalizado,WhatsApp,Envío de un cupón de descuento exclusivo por W...,Contactar al cliente por WhatsApp para ofrecer...,Enviarle un correo electrónico con recomendaci...,Realizar una llamada telefónica para ofrecerle...,Ofrecerle una visita comercial personalizada e...


In [51]:
import pprint

pprint.pprint(df_sample["estrategia_json"].iloc[0])

{'analisis_breve': 'El cliente con ID 1027461 está en el segmento En riesgo y '
                   'tiene una probabilidad estimada de churn del 68.9%. La '
                   'última compra fue hace 307 días y el total de compras fue '
                   'de 5 con un monto total comprado de S/ 1,970.18.',
 'canal_sugerido': 'WhatsApp',
 'estrategias': {'accion_1': 'Contactar al cliente por WhatsApp para ofrecerle '
                             'el descuento exclusivo.',
                 'accion_2': 'Enviar un correo electrónico personalizado al '
                             'cliente con el descuento exclusivo y una '
                             'invitación a visitar la tienda en línea.',
                 'accion_3': 'Realizar una llamada telefónica al cliente para '
                             'ofrecerle el descuento exclusivo y responder a '
                             'cualquier pregunta o duda que tenga.',
                 'accion_4': 'Enviar una carta de correo postal personal

## 11. Guardar resultados

In [52]:
OUTPUT_PATH = Path('../data/processed/clientes_recomendaciones_llm_sample.csv')

if not OUTPUT_PATH.parent.exists():
    OUTPUT_PATH = Path('/mnt/data/clientes_recomendaciones_llm_sample.csv')

df_estrategias.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Resultados guardados en: {OUTPUT_PATH}')

Resultados guardados en: ..\data\processed\clientes_recomendaciones_llm_sample.csv


## 12. Revisión rápida de consistencia

In [53]:
df_estrategias[[
    "cliente_id_new",
    "segmento",
    "score_churn",
    "prioridad_prompt",
    "prioridad",
    "canal_prompt",
    "canal_sugerido",
    "incentivo_recomendado",
    "estrategias.accion_1",
    "estrategias.accion_2",
    "estrategias.accion_3",
    "estrategias.accion_4"
]]

,cliente_id_new,segmento,score_churn,prioridad_prompt,prioridad,canal_prompt,canal_sugerido,incentivo_recomendado,estrategias.accion_1,estrategias.accion_2,estrategias.accion_3,estrategias.accion_4
0,1027461,En riesgo,0.688902,Alta - Contacto personalizado,Alta - Contacto personalizado,WhatsApp,WhatsApp,"Ofrecer un descuento exclusivo por WhatsApp, p...",Contactar al cliente por WhatsApp para ofrecer...,Enviar un correo electrónico personalizado al ...,Realizar una llamada telefónica al cliente par...,Enviar una carta de correo postal personalizad...
1,1089134,Estratégicos,0.144226,Baja - Mantener relación,Baja - Mantener relación,Email,Email,No aplicable,Enviar un correo electrónico personalizado al ...,Realizar una llamada telefónica al cliente par...,Enviar un correo electrónico personalizado al ...,Realizar una visita comercial al cliente en su...
2,1089187,Estratégicos,0.442597,Media - Monitorear comportamiento,Media - Monitorear comportamiento,WhatsApp,WhatsApp,Envío personalizado de un cupón de descuento p...,Contactarlo a través de WhatsApp para ofrecerl...,Enviarle un correo electrónico con información...,Realizar una llamada telefónica para ofrecerle...,Enviarle una notificación push de WhatsApp con...
3,1111859,Estratégicos,0.201553,Baja - Mantener relación,Baja - Mantener relación,Email,Email,Envío de ofertas especiales periódicas por cor...,Envío de una carta de agradecimiento personali...,Envío de una oferta especial por correo electr...,Envío de una oferta de regalo por correo elect...,Envío de una oferta de membresía por correo el...
4,1133970,En riesgo,0.744322,Alta - Contacto personalizado,Alta - Contacto personalizado,WhatsApp,WhatsApp,Envío de un cupón de descuento exclusivo por W...,Contactar al cliente por WhatsApp para ofrecer...,Enviarle un correo electrónico con recomendaci...,Realizar una llamada telefónica para ofrecerle...,Ofrecerle una visita comercial personalizada e...
